# Hacker News BigQuery Seed Generator Example

This notebook demonstrates how to generate binary forecasting questions for Hacker News post success using the BigQuery seed generator transform on the public Hacker News dataset.

In [18]:
%pip install lightningrod-ai python-dotenv

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

> **GCP credentials**: BigQuery requires GCP credentials. Set `GOOGLE_APPLICATION_CREDENTIALS` or use default application credentials. Optionally pass `project_id` to BigQuerySeedGenerator if needed.

In [19]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## BigQuery Seed Generator Configuration

The query fetches HN stories from the [public Hacker News dataset](https://console.cloud.google.com/marketplace/product/y-combinator/hacker-news) and computes labels in SQL at multiple thresholds (original: 100+ upvotes, front page, 50+ comments; balanced: 10+ upvotes, 5+ upvotes, 5+ comments) for varied prediction targets. These are embedded in the seed text for the QuestionAndLabelGenerator to use.

In [20]:
from lightningrod import BigQuerySeedGenerator

HN_QUERY = """
SELECT
  CONCAT(
    'Title: ', COALESCE(title, ''), '\\n\\nContent: ', COALESCE(REGEXP_REPLACE(COALESCE(text, ''), r'<[^>]*>', ''), COALESCE(url, '')),
    '\\n\\n[Computed labels - use for answers]: ',
    'Upvotes: ', score,
    'Comments: ', descendants
  ) AS content,
  TIMESTAMP_SECONDS(time) AS time
FROM `bigquery-public-data.hacker_news.full`
WHERE type = 'story' AND title IS NOT NULL AND (text IS NOT NULL OR url IS NOT NULL) AND score IS NOT NULL
ORDER BY time DESC
"""

seed_generator = BigQuerySeedGenerator(
    query=HN_QUERY,
    seed_text_column="content",
    date_column="time",
    max_rows=100,
)

## Build Pipeline

Use the exact values from the **Question Generator Instructions** section above for `instructions`, `examples`, and `bad_examples`.

In [ ]:
from lightningrod import (
    BinaryAnswerType,
    QuestionAndLabelGenerator,
    QuestionPipeline,
    QuestionRenderer,
)

INSTRUCTIONS = (
    "Generate binary forecasting questions about whether an HN post will succeed based on its title and content. "
    "Questions must be forward-looking (predict outcomes at time of posting) and verifiable from actual post performance. "
    "Focus on: upvote thresholds, front page reach, comment count. "
    "Predict verifiable outcomes (upvotes, front page, comments). Avoid factual/retrospective questions (topic, date, author) or content-attribute questions (technology, company). "
    "The seed includes a '[Computed labels - use for answers]' section with pre-computed Yes/No answers. Use these exact values for labels — do not infer or override them."
)

EXAMPLES = [
    "Will this Hacker News post receive 100 or more upvotes?",
    "Will this post reach the Hacker News front page (top 30)?",
    "Will this post receive 50 or more comments?",
    "Will this Hacker News post receive 10 or more upvotes?",
    "Will this post reach the Hacker News front page (5+ upvotes)?",
    "Will this post receive 5 or more comments?",
]

BAD_EXAMPLES = [
    "What topic does this post discuss? (factual, not forecasting)",
    "When was this post published? (retrospective, not forward-looking)",
    "Who is the author of this post? (author-based, not outcome-based)",
    "Does this post mention a specific technology? (technology-based, not success metric)",
    "Which company is mentioned in this post? (company-based, not outcome-based)",
]

answer_type = BinaryAnswerType()

question_generator = QuestionAndLabelGenerator(
    answer_type=answer_type,
    questions_per_seed=1,
    instructions=INSTRUCTIONS,
    examples=EXAMPLES,
    bad_examples=BAD_EXAMPLES,
)

pipeline = QuestionPipeline(
    seed_generator=seed_generator,
    question_generator=question_generator,
    renderer=QuestionRenderer(
        template="QUESTION: {question_text}\n\nHACKER NEWS POST:\n{seed_text}\n\nANSWER INSTRUCTIONS: {answer_instructions}",
        answer_type=answer_type
    ),
)

## Run the Pipeline

In [22]:
dataset = lr.transforms.run(pipeline, max_questions=20, name="HN BigQuery")

Output()

> Note: This can take a few minutes to complete processing.

## View Results

In [23]:
%pip install pandas

from IPython.display import clear_output
clear_output()

In [28]:
import pandas as pd

samples = dataset.download()
rows = dataset.flattened()
df = pd.DataFrame(rows)

print(f"Generated {dataset.num_rows} samples (%.1f%% valid)\n" % (dataset.valid_count() / dataset.num_rows * 100))

df[["question_text", "label", "label_confidence", "seed_creation_date", "prompt"]]

Generated 100 samples (100.0% valid)



,question_text,label,label_confidence,seed_creation_date,prompt
0,Will this Hacker News post receive 10 or more ...,None,1.0,2026-03-10T06:39:27+00:00,"[{'role': 'user', 'content': 'QUESTION: Will t..."
1,Will this Hacker News post receive 10 or more ...,None,1.0,2026-03-10T08:51:27+00:00,"[{'role': 'user', 'content': 'QUESTION: Will t..."
2,Will this Hacker News post receive 10 or more ...,None,1.0,2026-03-10T06:26:03+00:00,"[{'role': 'user', 'content': 'QUESTION: Will t..."
3,Will this Hacker News post receive 10 or more ...,None,1.0,2026-03-10T08:39:15+00:00,"[{'role': 'user', 'content': 'QUESTION: Will t..."
4,Will this Hacker News post receive 10 or more ...,None,1.0,2026-03-10T08:08:33+00:00,"[{'role': 'user', 'content': 'QUESTION: Will t..."
...,...,...,...,...,...
95,Will this Hacker News post receive 5 or more u...,None,1.0,2026-03-10T07:23:02+00:00,"[{'role': 'user', 'content': 'QUESTION: Will t..."
96,Will this Hacker News post receive 10 or more ...,None,1.0,2026-03-10T06:22:22+00:00,"[{'role': 'user', 'content': 'QUESTION: Will t..."
97,Will this Hacker News post receive 10 or more ...,None,1.0,2026-03-10T07:18:40+00:00,"[{'role': 'user', 'content': 'QUESTION: Will t..."
98,Will this Hacker News post receive 10 or more ...,None,1.0,2026-03-10T09:00:52+00:00,"[{'role': 'user', 'content': 'QUESTION: Will t..."
